# 04 — Evaluación Detallada V2

**Objetivo:** Analizar el rendimiento del modelo final bajo métricas de negocio y explicar sus decisiones con SHAP.

**Contenido:**
1. Análisis de Umbral de Decisión.
2. Curva Precision-Recall.
3. Análisis de Errores (Falsos Positivos/Negativos).
4. Interpretabilidad Global y Local con SHAP.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
from sklearn.metrics import precision_recall_curve, confusion_matrix, classification_report

model = joblib.load("../models/best_model.joblib")
X_test = pd.read_csv("../data/processed/X_test_v2.csv")
y_test = pd.read_csv("../data/processed/y_test_v2.csv")["target"]
y_probs = model.predict_proba(X_test)[:, 1]

## 1. Análisis de Umbral de Negocio (0.35)

In [ ]:
thresholds = [0.3, 0.35, 0.4, 0.5, 0.6]
print(f"{'Umbral':<10} | {'Precision':<10} | {'Recall':<10}")
for t in thresholds:
    preds = (y_probs >= t).astype(int)
    from sklearn.metrics import precision_score, recall_score
    print(f"{t:<10.2f} | {precision_score(y_test, preds):<10.4f} | {recall_score(y_test, preds):<10.4f}")

# Seleccionamos 0.35 para priorizar captura de leads
y_pred_final = (y_probs >= 0.35).astype(int)

## 2. Curva Precision-Recall

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, y_probs)
plt.plot(recall, precision, color='purple', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Curva Precision-Recall V2')
plt.show()

## 3. Interpretabilidad con SHAP (Global)

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary Plot (Beeswarm)
plt.title("SHAP Global: Impacto de Features en la Conversión")
shap.summary_plot(shap_values[1], X_test)

## 4. Análisis de Lead Individual (Waterfall)

In [ ]:
# Seleccionamos un lead aleatorio del test set
idx = 10
print(f"Probabilidad del modelo: {y_probs[idx]:.2%}")
shap.plots._waterfall.waterfall_legacy(explainer.expected_value[1], shap_values[1][idx], feature_names=X_test.columns)